In [4]:
import pandas as pd

df = pd.read_csv('2019_hh_trials.csv')
df.head(5)

,Unnamed: 0,cur_gameloop,joint_action,layout,layout_name,reward,score,state,time_elapsed,time_left,trial_id,player_0_is_human,player_1_is_human,player_0_id,player_1_id
0,0,0.0,"[[0, 0], [0, 0]]","['XXPXX', 'O 2O', 'X1 X', 'XDXSX']",cramped_room,0.0,0.0,"{""players"": [{""position"": [1, 2], ""orientation...",0.154,180.0,0,True,True,2,3
1,1,1.0,"[[0, 0], [0, 0]]","['XXPXX', 'O 2O', 'X1 X', 'XDXSX']",cramped_room,0.0,0.0,"{""players"": [{""position"": [1, 2], ""orientation...",0.312,180.0,0,True,True,2,3
2,2,2.0,"[[0, 0], [0, 0]]","['XXPXX', 'O 2O', 'X1 X', 'XDXSX']",cramped_room,0.0,0.0,"{""players"": [{""position"": [1, 2], ""orientation...",0.461,180.0,0,True,True,2,3
3,3,3.0,"[[0, 0], [0, 0]]","['XXPXX', 'O 2O', 'X1 X', 'XDXSX']",cramped_room,0.0,0.0,"{""players"": [{""position"": [1, 2], ""orientation...",0.614,179.0,0,True,True,2,3
4,4,4.0,"[[0, 0], [0, 0]]","['XXPXX', 'O 2O', 'X1 X', 'XDXSX']",cramped_room,0.0,0.0,"{""players"": [{""position"": [1, 2], ""orientation...",1.043,179.0,0,True,True,2,3


In [2]:
print(df.shape)

(119177, 15)


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119177 entries, 0 to 119176
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         119177 non-null  int64  
 1   cur_gameloop       119177 non-null  float64
 2   joint_action       119177 non-null  object 
 3   layout             119177 non-null  object 
 4   layout_name        119177 non-null  object 
 5   reward             119177 non-null  float64
 6   score              119177 non-null  float64
 7   state              119177 non-null  object 
 8   time_elapsed       119177 non-null  float64
 9   time_left          119177 non-null  float64
 10  trial_id           119177 non-null  int64  
 11  player_0_is_human  119177 non-null  bool   
 12  player_1_is_human  119177 non-null  bool   
 13  player_0_id        119177 non-null  int64  
 14  player_1_id        119177 non-null  int64  
dtypes: bool(2), float64(5), int64(4), object(4)
memory 

In [ ]:
# Types of layouts and data points

df.layout_name.value_counts()   # every layout seems to have equal number of data points more or less

layout_name
cramped_room             24005
coordination_ring        23996
asymmetric_advantages    23994
random3                  23759
random0                  23423
Name: count, dtype: int64

In [ ]:
# check if all the players are human
print(df.player_0_is_human.value_counts())
print(df.player_1_is_human.value_counts())  # looks like everyone is human

player_0_is_human
True    119177
Name: count, dtype: int64
player_1_is_human
True    119177
Name: count, dtype: int64


In [ ]:
df.layout[0]    # 1 and 2 are the players, X is walls, O is onion, P is pot to cook, D is dishes and finally S is the delivery area

"['XXPXX', 'O  2O', 'X1  X', 'XDXSX']"

In [15]:
df.state[0]

'{"players": [{"position": [1, 2], "orientation": [0, -1], "held_object": null}, {"position": [3, 1], "orientation": [0, -1], "held_object": null}], "objects": [], "bonus_orders": [], "all_orders": [{"ingredients": ["onion"]}, {"ingredients": ["onion", "onion"]}, {"ingredients": ["onion", "onion", "onion"]}, {"ingredients": ["tomato"]}, {"ingredients": ["tomato", "tomato"]}, {"ingredients": ["tomato", "tomato", "tomato"]}, {"ingredients": ["onion", "tomato"]}, {"ingredients": ["onion", "onion", "tomato"]}, {"ingredients": ["onion", "tomato", "tomato"]}], "timestep": 0}'

In [6]:
import sys
import numpy as np
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from imitation.preprocessing import load_csv_rows, build_trial_records

rows = load_csv_rows("2019_hh_trials.csv")
trials, report = build_trial_records(rows)

cramped = [t for t in trials if t.layout_name == "cramped_room"]

print("num cramped trials:", len(cramped))
lengths = np.array([t.num_steps for t in cramped], dtype=np.int32)
rewards = np.array([t.total_sparse_reward for t in cramped], dtype=np.float32)

if len(cramped) > 0:
    print("steps min/median/max:", lengths.min(), np.median(lengths), lengths.max())
    print("reward min/median/max:", rewards.min(), np.median(rewards), rewards.max())
    for q in [50, 70, 80, 90, 95]:
        print(
            f"p{q} steps={np.percentile(lengths, q):.1f}, "
            f"reward={np.percentile(rewards, q):.1f}"
        )

num cramped trials: 20
steps min/median/max: 1143 1204.0 1204
reward min/median/max: 0.0 87.5 120.0
p50 steps=1204.0, reward=87.5
p70 steps=1204.0, reward=91.5
p80 steps=1204.0, reward=95.0
p90 steps=1204.0, reward=110.0
p95 steps=1204.0, reward=110.5


In [8]:
import numpy as np
from imitation.preprocessing import load_csv_rows, build_trial_records

rows = load_csv_rows("2019_hh_trials.csv")
trials, report = build_trial_records(rows)

cramped = [t for t in trials if t.layout_name == "cramped_room"]

print("total cramped trials:", len(cramped))
print()

step_threshold = 1000
reward_thresholds = [0, 60, 70, 80, 90, 100, 110]

print(f"{'min_reward':>10} | {'kept':>4} | {'median_reward':>13} | {'max_reward':>10}")
print("-" * 50)

for r in reward_thresholds:
    kept = [
        t for t in cramped
        if t.num_steps >= step_threshold and t.total_sparse_reward >= r
    ]
    rewards = [t.total_sparse_reward for t in kept]
    median_reward = float(np.median(rewards)) if rewards else None
    max_reward = max(rewards) if rewards else None
    print(f"{r:>10} | {len(kept):>4} | {str(median_reward):>13} | {str(max_reward):>10}")

total cramped trials: 20

min_reward | kept | median_reward | max_reward
--------------------------------------------------
         0 |   20 |          87.5 |      120.0
        60 |   15 |          90.0 |      120.0
        70 |   14 |          90.0 |      120.0
        80 |   12 |          92.5 |      120.0
        90 |   10 |          95.0 |      120.0
       100 |    3 |         110.0 |      120.0
       110 |    3 |         110.0 |      120.0


In [2]:
import json
from pathlib import Path
import pandas as pd

json_folder = Path("trajectories")

summary = []
for jf in sorted(json_folder.glob("*.json")):
    with open(jf, "r", encoding="utf-8") as f:
        payload = json.load(f)

    traj = payload.get("trajectory", [])
    if not traj:
        continue

    first = traj[0]
    last = traj[-1]

    summary.append({
        "file": jf.name,
        "rows": len(traj),
        "layout_name": first.get("layout_name"),
        "trial_id": first.get("trial_id"),
        "final_score": last.get("score"),
        "start_gameloop": first.get("cur_gameloop"),
        "end_gameloop": last.get("cur_gameloop"),
    })

summary_df = pd.DataFrame(summary)
summary_df.sort_values(["layout_name", "rows"], ascending=[True, False])

,file,rows,layout_name,trial_id,final_score,start_gameloop,end_gameloop
0,game_0_9dc1db82_human-human_20260303_040417.json,867,cramped_room,None1772510627.2307289,100,1,867
3,game_1_1762ac00_human-human_20260310_022724.json,867,cramped_room,None1773109614.6305256,60,1,867
5,game_1_fe3ab84a_human-human_20260310_020127.json,867,cramped_room,None1773108057.0437903,140,1,867
6,game_2_6d7fc9f8_human-human_20260310_020203.json,867,cramped_room,None1773108093.0847054,100,1,867
13,game_4_390bedab_human-human_20260303_041319.json,867,cramped_room,None1772511169.491284,0,1,867
26,game_9_0fc86261_human-human_20260310_020048.json,867,cramped_room,None1773108017.9998088,100,1,867
11,game_3_895386f0_human-human_20260310_020331.json,866,cramped_room,None1773108181.8838787,60,1,866
14,game_4_901e21a8_human-human_20260310_022426.json,866,cramped_room,None1773109436.5005765,80,1,866
16,game_5_ecc1d24b_human-human_20260310_022805.json,866,cramped_room,None1773109655.1787488,60,1,866
17,game_5_fb369823_human-human_20260310_020241.json,866,cramped_room,None1773108131.2454607,80,1,866


In [3]:
import json
from pathlib import Path
import pandas as pd

json_folder = Path("trajectories")
existing_csv = Path("2019_hh_trials.csv")
output_csv = Path("2019_hh_trials_plus_local.csv")

required_cols = [
    "cur_gameloop",
    "joint_action",
    "layout",
    "layout_name",
    "reward",
    "score",
    "state",
    "time_elapsed",
    "time_left",
    "trial_id",
    "player_0_is_human",
    "player_1_is_human",
    "player_0_id",
    "player_1_id",
]

all_rows = []
json_files = sorted(json_folder.glob("*.json"))

for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
        payload = json.load(f)

    traj = payload.get("trajectory", [])
    if not traj:
        print(f"Skipping empty trajectory: {jf.name}")
        continue

    for step in traj:
        row = {col: step.get(col, None) for col in required_cols}

        # Match old CSV style:
        # - store state as JSON string
        # - store layout as Python-list-like string if needed
        if isinstance(row["state"], dict):
            row["state"] = json.dumps(row["state"])

        if isinstance(row["layout"], list):
            row["layout"] = repr(row["layout"])

        # joint_action in your files is already a string like "[[0, 0], [0, 0]]"
        # leave it as-is

        all_rows.append(row)

new_df = pd.DataFrame(all_rows, columns=required_cols)

print("JSON files found:", len(json_files))
print("New rows extracted:", len(new_df))
if not new_df.empty:
    print("Layouts:", new_df["layout_name"].value_counts().to_dict())
    print("Trials:", new_df["trial_id"].nunique())

# Load existing CSV
old_df = pd.read_csv(existing_csv)

# Drop old unnamed index column if present
unnamed = [c for c in old_df.columns if str(c).startswith("Unnamed:")]
if unnamed:
    old_df = old_df.drop(columns=unnamed)

# Make sure both dataframes have the same columns
for col in required_cols:
    if col not in old_df.columns:
        old_df[col] = None

old_df = old_df[required_cols]
new_df = new_df[required_cols]

# Optional duplicate protection:
# if you rerun this cell, it won't keep adding the same trial rows
combined = pd.concat([old_df, new_df], ignore_index=True)
combined = combined.drop_duplicates(subset=["trial_id", "cur_gameloop"], keep="last")

print("Old rows:", len(old_df))
print("Combined rows:", len(combined))
combined.to_csv(output_csv, index=False)
print(f"Saved to: {output_csv}")

JSON files found: 29
New rows extracted: 23927
Layouts: {'cramped_room': 23927}
Trials: 29
Old rows: 119177
Combined rows: 143104
Saved to: 2019_hh_trials_plus_local.csv


In [11]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from imitation.preprocessing import load_csv_rows, build_trial_records

rows = load_csv_rows("2019_hh_trials_plus_local.csv")
trials, report = build_trial_records(rows)

cramped = [t for t in trials if t.layout_name == "cramped_room"]

print("total cramped trials:", len(cramped))
print()

step_threshold = 800
reward_thresholds = [0, 60, 70, 80, 90, 100, 110]

print(f"{'min_reward':>10} | {'kept':>4} | {'median_reward':>13} | {'max_reward':>10}")
print("-" * 50)

for r in reward_thresholds:
    kept = [
        t for t in cramped
        if t.num_steps >= step_threshold and t.total_sparse_reward >= r
    ]
    rewards = [t.total_sparse_reward for t in kept]
    median_reward = float(np.median(rewards)) if rewards else None
    max_reward = max(rewards) if rewards else None
    print(f"{r:>10} | {len(kept):>4} | {str(median_reward):>13} | {str(max_reward):>10}")

total cramped trials: 49

min_reward | kept | median_reward | max_reward
--------------------------------------------------
         0 |   46 |          80.0 |      160.0
        60 |   39 |          85.0 |      160.0
        70 |   29 |          90.0 |      160.0
        80 |   27 |          95.0 |      160.0
        90 |   18 |         100.0 |      160.0
       100 |   11 |         110.0 |      160.0
       110 |    6 |         130.0 |      160.0


In [12]:
import json
from pathlib import Path
import pandas as pd

json_folder = Path("trajectories")
output_csv = Path("local_hh_trials.csv")

required_cols = [
    "cur_gameloop",
    "joint_action",
    "layout",
    "layout_name",
    "reward",
    "score",
    "state",
    "time_elapsed",
    "time_left",
    "trial_id",
    "player_0_is_human",
    "player_1_is_human",
    "player_0_id",
    "player_1_id",
]

all_rows = []
json_files = sorted(json_folder.glob("*.json"))

for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
        payload = json.load(f)

    traj = payload.get("trajectory", [])
    if not traj:
        print(f"Skipping empty trajectory: {jf.name}")
        continue

    for step in traj:
        row = {col: step.get(col, None) for col in required_cols}

        # Match old CSV style
        if isinstance(row["state"], dict):
            row["state"] = json.dumps(row["state"])

        if isinstance(row["layout"], list):
            row["layout"] = repr(row["layout"])

        all_rows.append(row)

df = pd.DataFrame(all_rows, columns=required_cols)

# Optional: avoid duplicate rows if two files share the same trial/timestep
df = df.drop_duplicates(subset=["trial_id", "cur_gameloop"], keep="last")

print("JSON files found:", len(json_files))
print("Rows written:", len(df))
if not df.empty:
    print("Unique trials:", df["trial_id"].nunique())
    print("Layouts:", df["layout_name"].value_counts().to_dict())

df.to_csv(output_csv, index=False)
print(f"Saved new CSV to: {output_csv}")

JSON files found: 29
Rows written: 23927
Unique trials: 29
Layouts: {'cramped_room': 23927}
Saved new CSV to: local_hh_trials.csv
